# v9 TPU SFT Candidate Policy

Self-contained Kaggle TPU notebook for v9 SFT. It executes an embedded in-memory trainer, spawns independent SFT members across TPU v5e-8 cores, uploads checkpoints every 30 epochs, and uploads final artifacts to `devaanshpa/orbit-wars-agent/v9/sft`. Defaults target the 2500-game both-sides dataset with 8 TPU members, 260 epochs, and 8192-row/pair batches.


In [ ]:
from getpass import getpass
from pathlib import Path
import os

os.environ.setdefault("PJRT_DEVICE", "TPU")
os.environ.setdefault("V9_DEVICE", "tpu")
os.environ.setdefault("V9_TPU_CORES", "8")

if not os.environ.get("HF_TOKEN"):
    token = getpass("HF_TOKEN: ").strip()
    if not token:
        raise RuntimeError("HF_TOKEN is required to download data and upload v9 artifacts")
    os.environ["HF_TOKEN"] = token

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("PJRT_DEVICE:", os.environ.get("PJRT_DEVICE"))
print("working directory:", Path.cwd())


In [ ]:
import importlib.util
import subprocess
import sys

missing = [pkg for pkg, module in (("huggingface-hub", "huggingface_hub"), ("matplotlib", "matplotlib")) if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is required in the TPU runtime")
if importlib.util.find_spec("torch_xla") is None:
    raise RuntimeError("torch_xla is required. Switch the Kaggle accelerator to TPU v5e-8 before running v9.")

import torch
import torch_xla.core.xla_model as xm
print("torch", torch.__version__)
print("xla device sample:", xm.xla_device())


In [ ]:
import sys
import types

V9_TRAINER_CODE = 'import argparse\nimport csv\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\n\nHF_REPO_ID = "devaanshpa/orbit-wars-agent"\nHF_REPO_TYPE = "model"\nSFT_REMOTE_PREFIX = "v9/sft"\nGRPO_REMOTE_PREFIX = "v9/grpo"\n\nMETADATA_COLS = {\n    "label",\n    "selected",\n    "outcome_weight",\n    "game_result",\n    "reward_margin",\n    "agent_reward",\n    "opponent_reward",\n    "selected_heuristic_rank",\n    "counterfactual_positive",\n    "counterfactual_reason",\n    "failure_overcommit",\n    "failure_missed_tactical",\n    "failure_missed_comet",\n    "failure_slow_expansion",\n    "turn_advantage",\n    "future_advantage_delta_5",\n    "future_advantage_delta_15",\n    "future_advantage_delta_30",\n    "future_production_delta_15",\n    "future_planet_delta_15",\n    "game_id",\n    "candidate_id",\n    "version",\n}\n\n\ndef parse_args():\n    parser = argparse.ArgumentParser(\n        description="Train v9 Orbit Wars candidate policies on TPU v5e-8."\n    )\n    parser.add_argument("--mode", choices=("sft", "grpo"), required=True)\n    parser.add_argument("--csv", default=os.environ.get("CANDIDATES_CSV", ""))\n    parser.add_argument(\n        "--data-remote-path",\n        default=os.environ.get("V9_DATA_REMOTE_PATH", ""),\n        help="Exact HF data path. If omitted, newest data/*/candidates_v9.csv or candidates_v7.csv is used.",\n    )\n    parser.add_argument("--sft-artifact", default=os.environ.get("V9_SFT_ARTIFACT", ""))\n    parser.add_argument(\n        "--sft-remote-path",\n        default=os.environ.get("V9_SFT_REMOTE_PATH", f"{SFT_REMOTE_PREFIX}/model_weights_v9_sft.json"),\n    )\n    parser.add_argument("--export-dir", default=os.environ.get("V9_EXPORT_DIR", "notebooks/v9/exports"))\n    parser.add_argument("--device", choices=("tpu", "cuda", "cpu", "auto"), default=os.environ.get("V9_DEVICE", "tpu"))\n    parser.add_argument("--tpu-cores", type=int, default=int(os.environ.get("V9_TPU_CORES", "8")))\n    parser.add_argument("--members", type=int, default=int(os.environ.get("V9_MEMBERS", "8")))\n    parser.add_argument("--epochs", type=int, default=int(os.environ.get("V9_EPOCHS", "0")))\n    parser.add_argument("--row-batch-size", type=int, default=int(os.environ.get("V9_ROW_BATCH_SIZE", "8192")))\n    parser.add_argument("--pair-batch-size", type=int, default=int(os.environ.get("V9_PAIR_BATCH_SIZE", "8192")))\n    parser.add_argument("--lr", type=float, default=float(os.environ.get("V9_LR", "0.00048")))\n    parser.add_argument("--weight-decay", type=float, default=float(os.environ.get("V9_WEIGHT_DECAY", "0.00025")))\n    parser.add_argument("--dropout", type=float, default=float(os.environ.get("V9_DROPOUT", "0.15")))\n    parser.add_argument("--bce-weight", type=float, default=float(os.environ.get("V9_BCE_WEIGHT", "0.62")))\n    parser.add_argument("--pair-weight", type=float, default=float(os.environ.get("V9_PAIR_WEIGHT", "0.52")))\n    parser.add_argument("--reward-weight", type=float, default=float(os.environ.get("V9_REWARD_WEIGHT", "0.78")))\n    parser.add_argument("--kl-weight", type=float, default=float(os.environ.get("V9_KL_WEIGHT", "0.075")))\n    parser.add_argument("--anchor-weight", type=float, default=float(os.environ.get("V9_ANCHOR_WEIGHT", "0.16")))\n    parser.add_argument("--patience", type=int, default=int(os.environ.get("V9_PATIENCE", "36")))\n    parser.add_argument("--checkpoint-every", type=int, default=int(os.environ.get("V9_CHECKPOINT_EVERY", "30")))\n    parser.add_argument("--eval-every", type=int, default=int(os.environ.get("V9_EVAL_EVERY", "1")))\n    parser.add_argument("--seed", type=int, default=int(os.environ.get("V9_SEED", "1909")))\n    parser.add_argument("--upload", action="store_true")\n    parser.add_argument("--hf-repo-id", default=os.environ.get("HF_REPO_ID", HF_REPO_ID))\n    parser.add_argument("--hf-repo-type", default=os.environ.get("HF_REPO_TYPE", HF_REPO_TYPE))\n    return parser.parse_args()\n\n\ndef load_dotenv(path=".env"):\n    env_path = Path(path)\n    if not env_path.exists():\n        return\n    for raw_line in env_path.read_text(encoding="utf-8").splitlines():\n        line = raw_line.strip()\n        if not line or line.startswith("#") or "=" not in line:\n            continue\n        key, value = line.split("=", 1)\n        key = key.strip()\n        value = value.strip().strip("\\"\'")\n        if key and key not in os.environ:\n            os.environ[key] = value\n\n\ndef hf_token():\n    load_dotenv()\n    return os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")\n\n\ndef download_training_csv(remote_path, repo_id, repo_type):\n    token = hf_token()\n    if not token:\n        raise RuntimeError("HF_TOKEN is required to download training data from Hugging Face.")\n    try:\n        from huggingface_hub import HfApi, hf_hub_download\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("Install huggingface_hub to download data.") from exc\n\n    if not remote_path:\n        api = HfApi(token=token)\n        files = api.list_repo_files(repo_id=repo_id, repo_type=repo_type)\n        candidates_v9 = sorted(\n            name for name in files if name.startswith("data/") and name.endswith("/candidates_v9.csv")\n        )\n        candidates_v7 = sorted(\n            name for name in files if name.startswith("data/") and name.endswith("/candidates_v7.csv")\n        )\n        candidates = candidates_v9 or candidates_v7\n        if not candidates:\n            raise FileNotFoundError("No data/*/candidates_v9.csv or candidates_v7.csv found in Hugging Face repo.")\n        remote_path = candidates[-1]\n\n    return Path(\n        hf_hub_download(\n            repo_id=repo_id,\n            repo_type=repo_type,\n            filename=remote_path,\n            token=token,\n        )\n    )\n\n\ndef download_sft_artifact(remote_path, repo_id, repo_type):\n    token = hf_token()\n    if not token:\n        raise RuntimeError("HF_TOKEN is required to download the SFT artifact from Hugging Face.")\n    try:\n        from huggingface_hub import hf_hub_download\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("Install huggingface_hub to download SFT artifacts.") from exc\n    return Path(hf_hub_download(repo_id=repo_id, repo_type=repo_type, filename=remote_path, token=token))\n\n\ndef upload_file(args, path, path_in_repo, commit_message):\n    if not args.upload:\n        return\n    token = hf_token()\n    if not token:\n        raise RuntimeError("HF_TOKEN is required for upload.")\n    try:\n        from huggingface_hub import HfApi\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("Install huggingface_hub to upload artifacts.") from exc\n    api = HfApi(token=token)\n    api.create_repo(repo_id=args.hf_repo_id, repo_type=args.hf_repo_type, exist_ok=True)\n    api.upload_file(\n        path_or_fileobj=str(path),\n        path_in_repo=path_in_repo,\n        repo_id=args.hf_repo_id,\n        repo_type=args.hf_repo_type,\n        commit_message=commit_message,\n    )\n    print(f"uploaded {path_in_repo}", flush=True)\n\n\ndef upload_folder(args, folder, path_in_repo, commit_message):\n    if not args.upload:\n        return\n    token = hf_token()\n    if not token:\n        raise RuntimeError("HF_TOKEN is required for upload.")\n    try:\n        from huggingface_hub import HfApi\n    except ModuleNotFoundError as exc:\n        raise RuntimeError("Install huggingface_hub to upload artifacts.") from exc\n    api = HfApi(token=token)\n    api.create_repo(repo_id=args.hf_repo_id, repo_type=args.hf_repo_type, exist_ok=True)\n    api.upload_folder(\n        folder_path=str(folder),\n        repo_id=args.hf_repo_id,\n        repo_type=args.hf_repo_type,\n        path_in_repo=path_in_repo,\n        commit_message=commit_message,\n    )\n    print(f"uploaded folder to {path_in_repo}", flush=True)\n\n\ndef resolve_local_inputs(args):\n    if args.csv:\n        csv_path = Path(args.csv)\n        if not csv_path.exists():\n            raise FileNotFoundError(f"Training CSV does not exist: {csv_path}")\n    else:\n        csv_path = download_training_csv(args.data_remote_path, args.hf_repo_id, args.hf_repo_type)\n    args.csv = str(csv_path)\n\n    if args.mode == "grpo":\n        if args.sft_artifact:\n            sft_path = Path(args.sft_artifact)\n            if not sft_path.exists():\n                raise FileNotFoundError(f"SFT artifact does not exist: {sft_path}")\n        else:\n            sft_path = download_sft_artifact(args.sft_remote_path, args.hf_repo_id, args.hf_repo_type)\n        args.sft_artifact = str(sft_path)\n    return args\n\n\ndef row_float(row, key, default=0.0):\n    try:\n        return float(row.get(key, default) or default)\n    except (TypeError, ValueError):\n        return float(default)\n\n\ndef read_rows(path):\n    with Path(path).open(newline="", encoding="utf-8") as f:\n        rows = list(csv.DictReader(f))\n    if not rows:\n        raise RuntimeError("Training CSV has no rows.")\n    feature_names = [name for name in rows[0] if name not in METADATA_COLS]\n    x_raw = [[row_float(row, name, 0.0) for name in feature_names] for row in rows]\n    labels = [max(0.0, min(1.0, row_float(row, "label", 0.0))) for row in rows]\n    selected = [row_float(row, "selected", 0.0) >= 0.5 for row in rows]\n    counterfactual = [row_float(row, "counterfactual_positive", 0.0) >= 0.5 for row in rows]\n    sample_weights = [max(0.05, row_float(row, "outcome_weight", 1.0)) for row in rows]\n    return rows, feature_names, x_raw, labels, selected, counterfactual, sample_weights\n\n\ndef split_by_game(rows, seed, validation_fraction=0.2):\n    games = sorted({row.get("game_id", "") for row in rows})\n    rng = random.Random(seed)\n    rng.shuffle(games)\n    valid_game_count = max(1, int(len(games) * validation_fraction)) if len(games) > 1 else 1\n    valid_games = set(games[:valid_game_count])\n    valid_indices = [i for i, row in enumerate(rows) if row.get("game_id", "") in valid_games]\n    valid_set = set(valid_indices)\n    train_indices = [i for i in range(len(rows)) if i not in valid_set] or valid_indices[:]\n    return train_indices, valid_indices, games, valid_games\n\n\ndef normalize_from_train(x_raw, train_indices):\n    train_raw = [x_raw[i] for i in train_indices]\n    means = [sum(col) / len(col) for col in zip(*train_raw)]\n    scales = []\n    for j, mean in enumerate(means):\n        var = sum((row[j] - mean) ** 2 for row in train_raw) / max(1, len(train_raw) - 1)\n        scales.append(max(1e-6, math.sqrt(var)))\n\n    def normalize(items):\n        return [[(row[j] - means[j]) / scales[j] for j in range(len(means))] for row in items]\n\n    return means, scales, normalize\n\n\ndef normalize_with_artifact(x_raw, feature_names, artifact):\n    member = artifact["members"][0] if artifact.get("members") else artifact\n    means = member.get("mean", {})\n    scales = member.get("scale", {})\n    normalized = []\n    for row in x_raw:\n        values = []\n        for name, value in zip(feature_names, row):\n            scale = float(scales.get(name, 1.0) or 1.0)\n            values.append((value - float(means.get(name, 0.0))) / scale)\n        normalized.append(values)\n    return normalized, means, scales\n\n\ndef build_groups(rows, indices):\n    grouped = defaultdict(list)\n    for raw_index in indices:\n        step = int(row_float(rows[raw_index], "step", 0.0))\n        grouped[(rows[raw_index].get("game_id", ""), step)].append(raw_index)\n    return [items for items in grouped.values() if len(items) >= 2]\n\n\ndef candidate_reward(row, label, selected, counterfactual):\n    reward = (label - 0.5) * 2.0\n    reward += max(-1.5, min(1.5, row_float(row, "future_advantage_delta_15", 0.0) / 80.0))\n    reward += max(-1.0, min(1.0, row_float(row, "future_advantage_delta_30", 0.0) / 120.0)) * 0.55\n    reward += max(-0.8, min(0.8, row_float(row, "future_production_delta_15", 0.0) / 8.0)) * 0.55\n    reward += max(-0.8, min(0.8, row_float(row, "future_planet_delta_15", 0.0) / 3.0)) * 0.45\n    reward += row_float(row, "game_result", 0.0) * 0.22\n    reward += max(-1.0, min(1.0, row_float(row, "reward_margin", 0.0) / 700.0)) * 0.18\n    if selected:\n        reward += 0.18\n    if counterfactual:\n        reward += 0.32\n    reward -= row_float(row, "failure_overcommit", 0.0) * 0.75\n    reward -= row_float(row, "failure_missed_tactical", 0.0) * 0.30\n    reward -= row_float(row, "failure_missed_comet", 0.0) * 0.30\n    reward -= row_float(row, "failure_slow_expansion", 0.0) * 0.25\n    return reward\n\n\ndef build_pairs(rows, labels, selected, counterfactual, groups, mode):\n    pairs = []\n    for group in groups:\n        if mode == "sft":\n            positives = [i for i in group if selected[i] or counterfactual[i] or labels[i] >= 0.55]\n            negatives = [i for i in group if labels[i] <= 0.35 and not counterfactual[i]]\n            positives.sort(key=lambda i: labels[i] + row_float(rows[i], "future_advantage_delta_15", 0.0) * 0.004, reverse=True)\n            negatives.sort(key=lambda i: row_float(rows[i], "heuristic_score_scaled", 0.0), reverse=True)\n        else:\n            rewards = [(i, candidate_reward(rows[i], labels[i], selected[i], counterfactual[i])) for i in group]\n            rewards.sort(key=lambda item: item[1], reverse=True)\n            positives = [i for i, _ in rewards[: max(1, min(3, len(rewards) // 3))]]\n            negatives = [i for i, _ in rewards[-max(1, min(5, len(rewards) // 2)) :]]\n        emitted = 0\n        for pos in positives:\n            for neg in negatives:\n                if pos == neg or emitted >= 10:\n                    break\n                gap = max(0.05, labels[pos] - labels[neg]) if mode == "sft" else 1.0\n                pairs.append((pos, neg, gap))\n                emitted += 1\n    return pairs\n\n\ndef grouped_metrics(rows, scores, positive_mask, indices):\n    groups = build_groups(rows, indices)\n    hits = 0\n    total = 0\n    ranks = []\n    for group in groups:\n        positives = [i for i in group if positive_mask[i]]\n        if not positives:\n            continue\n        ordered = sorted(group, key=lambda i: scores[i], reverse=True)\n        total += 1\n        if positive_mask[ordered[0]]:\n            hits += 1\n        ranks.append(min(ordered.index(i) + 1 for i in positives) / max(1, len(ordered)))\n    return {\n        "top1": hits / total if total else 0.0,\n        "rank_fraction": sum(ranks) / len(ranks) if ranks else 1.0,\n        "turns": total,\n    }\n\n\ndef sigmoid_prob(value):\n    value = max(-50.0, min(50.0, value))\n    return 1.0 / (1.0 + math.exp(-value))\n\n\ndef make_model_class(torch, nn):\n    class CandidatePolicy(nn.Module):\n        def __init__(self, feature_count, dropout):\n            super().__init__()\n            self.net = nn.Sequential(\n                nn.Linear(feature_count, 192),\n                nn.ReLU(),\n                nn.Dropout(dropout),\n                nn.Linear(192, 96),\n                nn.ReLU(),\n                nn.Dropout(dropout),\n                nn.Linear(96, 48),\n                nn.ReLU(),\n                nn.Linear(48, 1),\n            )\n\n        def forward(self, inputs):\n            return self.net(inputs).view(-1)\n\n    return CandidatePolicy\n\n\ndef layers_from_model(model, nn):\n    linear_layers = [module for module in model.net if isinstance(module, nn.Linear)]\n    layers = []\n    for index, layer in enumerate(linear_layers):\n        layers.append(\n            {\n                "weights": layer.weight.detach().cpu().tolist(),\n                "bias": layer.bias.detach().cpu().tolist(),\n                "activation": "relu" if index < len(linear_layers) - 1 else "linear",\n            }\n        )\n    return layers\n\n\ndef load_member_into_model(torch, model, member):\n    linear_layers = [module for module in model.net if module.__class__.__name__ == "Linear"]\n    for layer_module, layer_data in zip(linear_layers, member.get("layers", [])):\n        if len(layer_data.get("weights", [])) != layer_module.weight.shape[0]:\n            continue\n        layer_module.weight.data = torch.tensor(\n            layer_data["weights"], dtype=layer_module.weight.dtype, device=layer_module.weight.device\n        )\n        layer_module.bias.data = torch.tensor(\n            layer_data["bias"], dtype=layer_module.bias.dtype, device=layer_module.bias.device\n        )\n\n\ndef prepare_device(args, ordinal=0):\n    import torch\n\n    requested = args.device\n    if requested == "auto":\n        requested = "tpu" if os.environ.get("TPU_NAME") or os.environ.get("PJRT_DEVICE") == "TPU" else "cuda"\n    if requested == "tpu":\n        try:\n            import torch_xla.core.xla_model as xm\n        except ModuleNotFoundError as exc:\n            raise RuntimeError("torch_xla is required for --device tpu. Use a Kaggle TPU v5e-8 runtime.") from exc\n        return torch, xm.xla_device(), xm, True\n    if requested == "cuda" and torch.cuda.is_available():\n        return torch, torch.device(f"cuda:{ordinal % max(1, torch.cuda.device_count())}"), None, False\n    return torch, torch.device("cpu"), None, False\n\n\ndef optimizer_step(optimizer, xm):\n    if xm is None:\n        optimizer.step()\n    else:\n        xm.optimizer_step(optimizer, barrier=False)\n        xm.mark_step()\n\n\ndef metric_objective(metrics):\n    return (1.0 - metrics["top1"]) + 0.35 * metrics["rank_fraction"]\n\n\ndef save_member(path, payload):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")\n    return path\n\n\ndef train_one_sft_member(args, member_index, ordinal):\n    torch, device, xm, is_xla = prepare_device(args, ordinal)\n    from torch import nn\n    import torch.nn.functional as functional\n\n    rows, feature_names, x_raw, labels, selected, counterfactual, sample_weights = read_rows(args.csv)\n    train_indices, valid_indices, games, valid_games = split_by_game(rows, args.seed)\n    means, scales, normalize = normalize_from_train(x_raw, train_indices)\n    all_x = normalize(x_raw)\n    train_groups = build_groups(rows, train_indices)\n    valid_groups = build_groups(rows, valid_indices)\n    train_pairs = build_pairs(rows, labels, selected, counterfactual, train_groups, "sft")\n    positive_mask = [selected[i] or counterfactual[i] or labels[i] >= 0.55 for i in range(len(rows))]\n\n    member_seed = args.seed + member_index * 1009\n    random.seed(member_seed)\n    torch.manual_seed(member_seed)\n    CandidatePolicy = make_model_class(torch, nn)\n    model = CandidatePolicy(len(feature_names), args.dropout).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    epochs = args.epochs or 220\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))\n\n    x_all = torch.tensor(all_x, dtype=torch.float32, device=device)\n    y_all = torch.tensor(labels, dtype=torch.float32, device=device)\n    w_all = torch.tensor(sample_weights, dtype=torch.float32, device=device)\n    best_state = None\n    best_objective = float("inf")\n    stale = 0\n    history = []\n    start = time.time()\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        random.shuffle(train_indices)\n        random.shuffle(train_pairs)\n        total_loss = 0.0\n        batches = 0\n        pair_cursor = 0\n        for offset in range(0, len(train_indices), args.row_batch_size):\n            batch = train_indices[offset : offset + args.row_batch_size]\n            if len(batch) < 2:\n                continue\n            optimizer.zero_grad(set_to_none=True)\n            idx = torch.tensor(batch, dtype=torch.long, device=device)\n            logits = model(x_all[idx])\n            bce = functional.binary_cross_entropy_with_logits(logits, y_all[idx], weight=w_all[idx])\n            loss = args.bce_weight * bce\n            if train_pairs:\n                pair_batch = train_pairs[pair_cursor : pair_cursor + args.pair_batch_size]\n                pair_cursor = (pair_cursor + args.pair_batch_size) % max(1, len(train_pairs))\n                if pair_batch:\n                    pos_idx = torch.tensor([p[0] for p in pair_batch], dtype=torch.long, device=device)\n                    neg_idx = torch.tensor([p[1] for p in pair_batch], dtype=torch.long, device=device)\n                    pair_w = torch.tensor([p[2] for p in pair_batch], dtype=torch.float32, device=device)\n                    pair_loss = (functional.softplus(-(model(x_all[pos_idx]) - model(x_all[neg_idx]))) * pair_w).mean()\n                    loss = loss + args.pair_weight * pair_loss\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            optimizer_step(optimizer, xm)\n            total_loss += float(loss.detach().cpu())\n            batches += 1\n        scheduler.step()\n\n        if epoch % args.eval_every == 0 or epoch == 1 or epoch == epochs:\n            model.eval()\n            with torch.no_grad():\n                logits_all = model(x_all).detach().cpu().tolist()\n            scores = [sigmoid_prob(value) for value in logits_all]\n            valid_metrics = grouped_metrics(rows, scores, positive_mask, valid_indices)\n            objective = metric_objective(valid_metrics)\n            item = {\n                "epoch": epoch,\n                "train_loss": total_loss / max(1, batches),\n                "valid_turn_top1": valid_metrics["top1"],\n                "valid_rank_fraction": valid_metrics["rank_fraction"],\n                "objective": objective,\n                "lr": scheduler.get_last_lr()[0],\n                "elapsed_seconds": round(time.time() - start, 2),\n            }\n            history.append(item)\n            print(\n                f"v9 sft member={member_index} epoch={epoch:03d} "\n                f"loss={item[\'train_loss\']:.5f} top1={item[\'valid_turn_top1\']:.4f} "\n                f"rank={item[\'valid_rank_fraction\']:.4f} device={device}",\n                flush=True,\n            )\n            if args.checkpoint_every > 0 and epoch % args.checkpoint_every == 0:\n                checkpoint = {\n                    "version": "v9_sft_checkpoint",\n                    "member_index": member_index,\n                    "epoch": epoch,\n                    "latest_metrics": item,\n                    "history": history,\n                    "member": {\n                        "version": "v9_sft",\n                        "model_type": "mlp_relu_candidate_ranker",\n                        "features": feature_names,\n                        "mean": dict(zip(feature_names, means)),\n                        "scale": dict(zip(feature_names, scales)),\n                        "layers": layers_from_model(model, nn),\n                        "activation": "relu",\n                        "score_scale": 235.0,\n                    },\n                }\n                path = save_member(\n                    Path(args.export_dir) / "sft" / "checkpoints" / f"sft_member_{member_index:02d}_epoch_{epoch:03d}.json",\n                    checkpoint,\n                )\n                upload_file(args, path, f"{SFT_REMOTE_PREFIX}/checkpoints/{path.name}", f"Upload v9 SFT member {member_index} epoch {epoch}")\n            if objective + 1e-5 < best_objective:\n                best_objective = objective\n                best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n                stale = 0\n            else:\n                stale += 1\n                if stale >= args.patience:\n                    print(f"v9 sft member={member_index} early_stop epoch={epoch}", flush=True)\n                    break\n\n    if best_state is not None:\n        model.load_state_dict(best_state)\n    model.eval()\n    payload = {\n        "version": "v9_sft",\n        "model_type": "mlp_relu_candidate_ranker",\n        "features": feature_names,\n        "mean": dict(zip(feature_names, means)),\n        "scale": dict(zip(feature_names, scales)),\n        "layers": layers_from_model(model, nn),\n        "activation": "relu",\n        "score_scale": 235.0,\n        "member_index": member_index,\n        "history": history,\n        "best_validation_objective": best_objective,\n    }\n    path = save_member(Path(args.export_dir) / "sft" / "members" / f"member_{member_index:02d}.json", payload)\n    print(f"saved sft member {member_index}: {path}", flush=True)\n\n\ndef train_one_grpo_member(args, member_index, ordinal):\n    torch, device, xm, is_xla = prepare_device(args, ordinal)\n    from torch import nn\n    import torch.nn.functional as functional\n\n    sft_artifact = json.loads(Path(args.sft_artifact).read_text(encoding="utf-8"))\n    rows, feature_names, x_raw, labels, selected, counterfactual, sample_weights = read_rows(args.csv)\n    all_x, means, scales = normalize_with_artifact(x_raw, feature_names, sft_artifact)\n    train_indices, valid_indices, games, valid_games = split_by_game(rows, args.seed)\n    train_groups = build_groups(rows, train_indices)\n    train_pairs = build_pairs(rows, labels, selected, counterfactual, train_groups, "grpo")\n    rewards = [candidate_reward(row, labels[i], selected[i], counterfactual[i]) for i, row in enumerate(rows)]\n    reward_targets = [sigmoid_prob(reward) for reward in rewards]\n    positive_mask = [selected[i] or counterfactual[i] or labels[i] >= 0.55 for i in range(len(rows))]\n\n    member_seed = args.seed + 5000 + member_index * 997\n    random.seed(member_seed)\n    torch.manual_seed(member_seed)\n    CandidatePolicy = make_model_class(torch, nn)\n    model = CandidatePolicy(len(feature_names), args.dropout).to(device)\n    ref_model = CandidatePolicy(len(feature_names), args.dropout).to(device)\n    base_members = sft_artifact.get("members", []) or [sft_artifact]\n    load_member_into_model(torch, model, base_members[member_index % len(base_members)])\n    load_member_into_model(torch, ref_model, base_members[member_index % len(base_members)])\n    ref_model.eval()\n    for param in ref_model.parameters():\n        param.requires_grad_(False)\n\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    epochs = args.epochs or 140\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))\n    x_all = torch.tensor(all_x, dtype=torch.float32, device=device)\n    reward_all = torch.tensor(reward_targets, dtype=torch.float32, device=device)\n    label_all = torch.tensor(labels, dtype=torch.float32, device=device)\n    best_state = None\n    best_objective = float("inf")\n    stale = 0\n    history = []\n    start = time.time()\n\n    for epoch in range(1, epochs + 1):\n        model.train()\n        random.shuffle(train_indices)\n        random.shuffle(train_pairs)\n        total_loss = 0.0\n        batches = 0\n        pair_cursor = 0\n        for offset in range(0, len(train_indices), args.row_batch_size):\n            batch = train_indices[offset : offset + args.row_batch_size]\n            if len(batch) < 2:\n                continue\n            optimizer.zero_grad(set_to_none=True)\n            idx = torch.tensor(batch, dtype=torch.long, device=device)\n            logits = model(x_all[idx])\n            with torch.no_grad():\n                ref_logits = ref_model(x_all[idx])\n            reward_loss = functional.binary_cross_entropy_with_logits(logits, reward_all[idx])\n            anchor_loss = functional.binary_cross_entropy_with_logits(logits, label_all[idx])\n            kl_proxy = ((logits - ref_logits) ** 2).mean()\n            loss = args.reward_weight * reward_loss + args.anchor_weight * anchor_loss + args.kl_weight * kl_proxy\n            if train_pairs:\n                pair_batch = train_pairs[pair_cursor : pair_cursor + args.pair_batch_size]\n                pair_cursor = (pair_cursor + args.pair_batch_size) % max(1, len(train_pairs))\n                if pair_batch:\n                    pos_idx = torch.tensor([p[0] for p in pair_batch], dtype=torch.long, device=device)\n                    neg_idx = torch.tensor([p[1] for p in pair_batch], dtype=torch.long, device=device)\n                    pair_loss = functional.softplus(-(model(x_all[pos_idx]) - model(x_all[neg_idx]))).mean()\n                    loss = loss + args.pair_weight * pair_loss\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.8)\n            optimizer_step(optimizer, xm)\n            total_loss += float(loss.detach().cpu())\n            batches += 1\n        scheduler.step()\n\n        if epoch % args.eval_every == 0 or epoch == 1 or epoch == epochs:\n            model.eval()\n            with torch.no_grad():\n                logits_all = model(x_all).detach().cpu().tolist()\n            scores = [sigmoid_prob(value) for value in logits_all]\n            valid_metrics = grouped_metrics(rows, scores, positive_mask, valid_indices)\n            objective = metric_objective(valid_metrics)\n            item = {\n                "epoch": epoch,\n                "train_loss": total_loss / max(1, batches),\n                "valid_turn_top1": valid_metrics["top1"],\n                "valid_rank_fraction": valid_metrics["rank_fraction"],\n                "objective": objective,\n                "lr": scheduler.get_last_lr()[0],\n                "elapsed_seconds": round(time.time() - start, 2),\n            }\n            history.append(item)\n            print(\n                f"v9 grpo member={member_index} epoch={epoch:03d} "\n                f"loss={item[\'train_loss\']:.5f} top1={item[\'valid_turn_top1\']:.4f} "\n                f"rank={item[\'valid_rank_fraction\']:.4f} device={device}",\n                flush=True,\n            )\n            if args.checkpoint_every > 0 and epoch % args.checkpoint_every == 0:\n                checkpoint = {\n                    "version": "v9_grpo_checkpoint",\n                    "member_index": member_index,\n                    "epoch": epoch,\n                    "latest_metrics": item,\n                    "history": history,\n                    "member": {\n                        "version": "v9_grpo",\n                        "model_type": "mlp_relu_candidate_ranker",\n                        "features": feature_names,\n                        "mean": means,\n                        "scale": scales,\n                        "layers": layers_from_model(model, nn),\n                        "activation": "relu",\n                        "score_scale": 260.0,\n                    },\n                }\n                path = save_member(\n                    Path(args.export_dir) / "grpo" / "checkpoints" / f"grpo_member_{member_index:02d}_epoch_{epoch:03d}.json",\n                    checkpoint,\n                )\n                upload_file(args, path, f"{GRPO_REMOTE_PREFIX}/checkpoints/{path.name}", f"Upload v9 GRPO member {member_index} epoch {epoch}")\n            if objective + 1e-5 < best_objective:\n                best_objective = objective\n                best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}\n                stale = 0\n            else:\n                stale += 1\n                if stale >= args.patience:\n                    print(f"v9 grpo member={member_index} early_stop epoch={epoch}", flush=True)\n                    break\n\n    if best_state is not None:\n        model.load_state_dict(best_state)\n    payload = {\n        "version": "v9_grpo",\n        "model_type": "mlp_relu_candidate_ranker",\n        "features": feature_names,\n        "mean": means,\n        "scale": scales,\n        "layers": layers_from_model(model, nn),\n        "activation": "relu",\n        "score_scale": 260.0,\n        "member_index": member_index,\n        "history": history,\n        "best_validation_objective": best_objective,\n    }\n    path = save_member(Path(args.export_dir) / "grpo" / "members" / f"member_{member_index:02d}.json", payload)\n    print(f"saved grpo member {member_index}: {path}", flush=True)\n\n\ndef worker_main(ordinal, args_dict):\n    args = argparse.Namespace(**args_dict)\n    world_size = max(1, int(args.world_size))\n    for member_index in range(ordinal, args.members, world_size):\n        if args.mode == "sft":\n            train_one_sft_member(args, member_index, ordinal)\n        else:\n            train_one_grpo_member(args, member_index, ordinal)\n\n\ndef aggregate_artifact(args):\n    mode_dir = Path(args.export_dir) / args.mode\n    member_files = sorted((mode_dir / "members").glob("member_*.json"))\n    if not member_files:\n        raise RuntimeError(f"No member files found in {mode_dir / \'members\'}")\n    members = [json.loads(path.read_text(encoding="utf-8")) for path in member_files]\n    histories = [{"member_index": member.get("member_index"), "history": member.pop("history", [])} for member in members]\n    bests = [member.pop("best_validation_objective", None) for member in members]\n    version = f"v9_{args.mode}"\n    score_scale = 235.0 if args.mode == "sft" else 260.0\n    blend = 0.30 if args.mode == "sft" else 0.42\n    artifact = {\n        "version": version,\n        "created_at": int(time.time()),\n        "source_csv": args.csv,\n        "source_sft_artifact": args.sft_artifact if args.mode == "grpo" else "",\n        "model_type": "ensemble_mlp_relu_candidate_ranker",\n        "members": members,\n        "blend": blend,\n        "score_scale": score_scale,\n        "metrics": {\n            "members": len(members),\n            "best_validation_objectives": bests,\n            "device": args.device,\n            "tpu_cores": args.tpu_cores,\n            "row_batch_size": args.row_batch_size,\n            "pair_batch_size": args.pair_batch_size,\n        },\n    }\n    mode_dir.mkdir(parents=True, exist_ok=True)\n    weights_name = f"model_weights_v9_{args.mode}.json"\n    metrics_name = f"metrics_v9_{args.mode}.json"\n    history_name = f"training_history_v9_{args.mode}.json"\n    (mode_dir / weights_name).write_text(json.dumps(artifact, indent=2, sort_keys=True), encoding="utf-8")\n    (mode_dir / metrics_name).write_text(json.dumps(artifact["metrics"], indent=2, sort_keys=True), encoding="utf-8")\n    (mode_dir / history_name).write_text(json.dumps(histories, indent=2, sort_keys=True), encoding="utf-8")\n    print(json.dumps(artifact["metrics"], indent=2, sort_keys=True), flush=True)\n    print(f"saved v9 {args.mode} artifact: {mode_dir / weights_name}", flush=True)\n    remote_prefix = SFT_REMOTE_PREFIX if args.mode == "sft" else GRPO_REMOTE_PREFIX\n    upload_folder(args, mode_dir, remote_prefix, f"Upload v9 {args.mode.upper()} TPU artifacts")\n\n\ndef run_training(args):\n    args = resolve_local_inputs(args)\n    default_epochs = 260 if args.mode == "sft" else 180\n    if args.epochs <= 0:\n        args.epochs = default_epochs\n    export_dir = Path(args.export_dir)\n    export_dir.mkdir(parents=True, exist_ok=True)\n    if args.device == "auto" and (os.environ.get("TPU_NAME") or os.environ.get("PJRT_DEVICE") == "TPU"):\n        args.device = "tpu"\n    args.world_size = min(max(1, args.members), max(1, args.tpu_cores if args.device == "tpu" else 1))\n\n    print(\n        json.dumps(\n            {\n                "mode": args.mode,\n                "csv": args.csv,\n                "sft_artifact": args.sft_artifact,\n                "device": args.device,\n                "world_size": args.world_size,\n                "members": args.members,\n                "epochs": args.epochs,\n                "row_batch_size": args.row_batch_size,\n                "pair_batch_size": args.pair_batch_size,\n                "export_dir": args.export_dir,\n                "upload": args.upload,\n            },\n            indent=2,\n        ),\n        flush=True,\n    )\n\n    if args.device == "tpu":\n        os.environ.setdefault("PJRT_DEVICE", "TPU")\n        try:\n            import torch_xla.distributed.xla_multiprocessing as xmp\n        except ModuleNotFoundError as exc:\n            raise RuntimeError("torch_xla is required for TPU training. Enable a Kaggle TPU v5e-8 runtime.") from exc\n        xmp.spawn(worker_main, args=(vars(args),), nprocs=args.world_size, start_method="fork")\n    else:\n        worker_main(0, vars(args))\n    aggregate_artifact(args)\n\n\ndef main():\n    run_training(parse_args())\n\n\nif __name__ == "__main__":\n    main()\n'

v9_trainer = types.ModuleType("inline_v9_tpu")
v9_trainer.__file__ = "<inline_v9_tpu>"
sys.modules["inline_v9_tpu"] = v9_trainer
exec(V9_TRAINER_CODE, v9_trainer.__dict__)
print("Loaded inline v9 TPU trainer; no companion .py file is required.")


In [ ]:
import argparse
import os
from pathlib import Path

def env_int(name, default):
    return int(os.environ.get(name, str(default)))

def env_float(name, default):
    return float(os.environ.get(name, str(default)))

export_dir = Path(os.environ.get("V9_SFT_EXPORT_DIR", str(Path.cwd() / "v9_exports")))
args = argparse.Namespace(
    mode="sft",
    csv=os.environ.get("CANDIDATES_CSV", ""),
    data_remote_path=os.environ.get("V9_DATA_REMOTE_PATH", ""),
    sft_artifact=os.environ.get("V9_SFT_ARTIFACT", ""),
    sft_remote_path=os.environ.get("V9_SFT_REMOTE_PATH", f"{v9_trainer.SFT_REMOTE_PREFIX}/model_weights_v9_sft.json"),
    export_dir=str(export_dir),
    device=os.environ.get("V9_DEVICE", "tpu"),
    tpu_cores=env_int("V9_TPU_CORES", 8),
    members=env_int("V9_SFT_MEMBERS", env_int("V9_MEMBERS", 8)),
    epochs=env_int("V9_SFT_EPOCHS", 260),
    row_batch_size=env_int("V9_SFT_ROW_BATCH_SIZE", env_int("V9_ROW_BATCH_SIZE", 8192)),
    pair_batch_size=env_int("V9_SFT_PAIR_BATCH_SIZE", env_int("V9_PAIR_BATCH_SIZE", 8192)),
    lr=env_float("V9_SFT_LR", env_float("V9_LR", 0.00048)),
    weight_decay=env_float("V9_SFT_WEIGHT_DECAY", env_float("V9_WEIGHT_DECAY", 0.00025)),
    dropout=env_float("V9_SFT_DROPOUT", env_float("V9_DROPOUT", 0.15)),
    bce_weight=env_float("V9_SFT_BCE_WEIGHT", env_float("V9_BCE_WEIGHT", 0.62)),
    pair_weight=env_float("V9_SFT_PAIR_WEIGHT", env_float("V9_PAIR_WEIGHT", 0.52)),
    reward_weight=env_float("V9_REWARD_WEIGHT", 0.78),
    kl_weight=env_float("V9_KL_WEIGHT", 0.075),
    anchor_weight=env_float("V9_ANCHOR_WEIGHT", 0.16),
    patience=env_int("V9_SFT_PATIENCE", env_int("V9_PATIENCE", 36)),
    checkpoint_every=env_int("V9_SFT_CHECKPOINT_EVERY", env_int("V9_CHECKPOINT_EVERY", 30)),
    eval_every=env_int("V9_EVAL_EVERY", 1),
    seed=env_int("V9_SEED", 1909),
    upload=os.environ.get("V9_SFT_UPLOAD", "1") != "0",
    hf_repo_id=os.environ.get("HF_REPO_ID", v9_trainer.HF_REPO_ID),
    hf_repo_type=os.environ.get("HF_REPO_TYPE", v9_trainer.HF_REPO_TYPE),
)
print({
    "mode": args.mode,
    "device": args.device,
    "tpu_cores": args.tpu_cores,
    "members": args.members,
    "epochs": args.epochs,
    "export_dir": args.export_dir,
    "upload": args.upload,
})
v9_trainer.run_training(args)


In [ ]:
from pathlib import Path
import json

export_dir = Path(os.environ.get("V9_SFT_EXPORT_DIR", str(Path.cwd() / "v9_exports"))) / "sft"
metrics_path = export_dir / "metrics_v9_sft.json"
weights_path = export_dir / "model_weights_v9_sft.json"
print("weights:", weights_path)
if metrics_path.exists():
    print(json.dumps(json.loads(metrics_path.read_text(encoding="utf-8")), indent=2, sort_keys=True))
else:
    print("metrics not found yet:", metrics_path)
